## Filter and convert to h5

Open up an emitter set, inspect some of the histograms, apply filters and resave as one large h5.

(it takes a few minutes to open up the entire collection of h5s, so filtering obviously bad fits and saving as a single h5 speeds up the loading a lot for the other notebooks)

In [1]:
import os
import decode
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

c:\Users\bnort\miniconda3\envs\decode_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
start_iter = 0 
end_iter = 1069
frames_per_iter = 250

out_path = r'D:\Janelia_slm_data\processed\Janelia PALM RUN2'
out_csv = os.path.join(out_path, f'emitters_{start_iter}_{end_iter}_filtered.csv')
h5s_path = os.path.join(out_path, f'processed-2025-06-27')

# comment back in if emitters were saved to a single file defined by out_csv

#emitters = decode.EmitterSet.load(out_csv)
#print(emitters)

In [3]:
out_h5 = os.path.join(out_path, f'emitters_{start_iter}_{end_iter}.h5')
print(out_h5)

D:\Janelia_slm_data\processed\Janelia PALM RUN2\emitters_0_1069.h5


In [4]:
emitters=decode.EmitterSet.load(out_h5)
print(emitters)

EmitterSet
::num emitters: 270211344
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 534249
::spanned volume: [-5.9900337e-01 -7.0383626e-01 -1.3020839e+03] - [ 799.6174   799.60876 1261.7509 ]


## Load sequence of iter results

Hess lab data is saved as hundreds to thousands of tif stacks representing sets of 250 frames.  We process the data iter by iter and save an H5 for each iter.  To read a back we need to loop through the iters reading each one. 

In [5]:
# create empty emitter set for loading
emitters = None
for i in tqdm(range(start_iter, end_iter), desc="Loading iterations"):
    #print(os.path.join(h5s_path, f'iter_{i}.h5'))
    out_h5 = os.path.join(h5s_path, f'iter_{i}.h5')
    if emitters is None:
        emitters = decode.EmitterSet.load(out_h5)
        emitters.frame_ix += i * frames_per_iter
    else:
        emitters_ = decode.EmitterSet.load(out_h5)
        #emitters_.frame_ix += i * frames_per_iter
        emitters = decode.EmitterSet.cat([emitters, emitters_])


Loading iterations: 100%|██████████| 1069/1069 [34:27<00:00,  1.93s/it]


In [6]:
emitters.frame_ix.min(), emitters.frame_ix.max()

(tensor(0), tensor(267249))

In [7]:
1069*250

267250

In [8]:
out_h5 = os.path.join(out_path, f'emitters_{start_iter}_{end_iter}.h5')
print(f'saving {out_h5}')
emitters.save(out_h5)

saving D:\Janelia_slm_data\processed\Janelia PALM RUN2\emitters_0_1069.h5


In [ ]:
sub_sample = 50

plt.figure(figsize=(18,4))
plt.subplot(131)
sns.distplot(emitters.xyz_sig_nm[::sub_sample, 0].numpy())
plt.xlabel('Sigma Estimate in X (nm)')

plt.subplot(132)
sns.distplot(emitters.xyz_sig_nm[::sub_sample, 1].numpy())
plt.xlabel('Sigma Estimate in Y (nm)')

plt.subplot(133)
sns.distplot(emitters.xyz_sig_nm[::sub_sample, 2].numpy())
plt.xlabel('Sigma Estimate in Z (nm)')

plt.show()

In [ ]:
print(emitters)
sigma_x_high_threshold = 50
sigma_y_high_threshold = 50
sigma_z_high_threshold = 100
prob_threshold = 0.5
coord_limit=((500,700),(100,200))
coord_limit=((0,500),(0,700))

print("0,1", coord_limit[0][0], coord_limit[0][1])

emitters_filtered = emitters[
    (emitters.xyz_sig_nm[:, 0] <= sigma_x_high_threshold)
    * (emitters.xyz_sig_nm[:, 1] <= sigma_x_high_threshold)
    * (emitters.xyz_sig_nm[:, 2] <= sigma_z_high_threshold)
    * (emitters.prob >= prob_threshold)
    ]

print(emitters_filtered)



In [ ]:
plt.figure(figsize=(18,4))
plt.subplot(131)
sns.distplot(emitters_filtered.xyz_sig_nm[:, 0].numpy())
plt.xlabel('Sigma Estimate in X (nm)')

plt.subplot(132)
sns.distplot(emitters_filtered.xyz_sig_nm[:, 1].numpy())
plt.xlabel('Sigma Estimate in Y (nm)')

plt.subplot(133)
sns.distplot(emitters_filtered.xyz_sig_nm[:, 2].numpy())
plt.xlabel('Sigma Estimate in Z (nm)')

plt.show()

In [ ]:
emitters_filtered.frame_ix.min(), emitters_filtered.frame_ix.max(), 250*300
emitters_filtered.frame_ix = emitters_filtered.frame_ix - emitters_filtered.frame_ix.min()
emitters_filtered.frame_ix.min(), emitters_filtered.frame_ix.max(), 250*300


In [ ]:
out_csv = os.path.join(out_path, f'emitters_{start_iter}_{end_iter}_filtered_{sigma_x_high_threshold}_{sigma_y_high_threshold}_{sigma_z_high_threshold}.csv')
print(f'saving {out_csv}')
emitters_filtered.save(out_csv)